# 🎬 Character Voice Aging System MVP — Qwen3-TTS

Demonstrates how a character's voice evolves across 4 distinct life stages for 8 iconic original character archetypes. Each character gets a complete audio life story.

| Character | Archetype Inspiration | Personality Arc |
|---|---|---|
| Marcus Vale | The Genius Inventor | Brash youth to weary wisdom |
| Sam Cole | The Young Hero | Scared kid to veteran mentor |
| Elara Vex | The Dark Vigilante | Raw grief to earned stillness |
| Kael Ryn | The Chosen Rebel | Impulsive rebel to transcendent calm |
| Viktor Cross | The Super Soldier | Idealistic soldier to haunted survivor |
| Diana Fortes | The Warrior Goddess | Clear certainty to ancient love |
| Walter Marsh | The Fallen Everyman | Mild-mannered to cold monster |
| James Holt | The Suave Operative | Cocky spy to weary authority |

**Age Stages:** Youth (16-22), Prime (28-36), Middle (46-55), Elder (65-78)

> **Note:** All characters are original fictional creations inspired by popular archetypes. We use VoiceDesign (text-prompt → synthetic voice) — no real actor voices are cloned.

In [ ]:
!pip install -q qwen-tts soundfile
import os, gc, numpy as np, soundfile as sf, torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUTPUT_DIR = "/content/character_aging"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Setup complete. Output dir: {OUTPUT_DIR}")

In [ ]:
CHARACTERS = {
    "marcus_vale": {
        "name": "Marcus Vale",
        "char_id": "marcus_vale",
        "archetype_note": "The Genius Inventor — brilliant, sarcastic engineer",
        "personality_core": "a brilliant, fast-talking, sarcastic genius engineer — confident bordering on arrogant, with a sharp wit and a heart he tries to hide",
        "ages": {
            "youth": {
                "label": "Youth (17)",
                "voice_modifier": "teenage overconfidence masking insecurity, rapid-fire speech, slightly cracking voice, reckless energy",
                "line": "I've already rebuilt the engine three times. The fourth version is the one that won't explode. Probably."
            },
            "prime": {
                "label": "Prime (32)",
                "voice_modifier": "full resonant confidence, quick wit perfectly controlled, charismatic and sharp, every word deliberate",
                "line": "I've done the math. Twice. Then I did it a third time because I don't trust math that agrees with me too easily."
            },
            "middle": {
                "label": "Middle (48)",
                "voice_modifier": "still sharp but warmer, slight pauses that weren't there before, protective undertone, depth replacing speed",
                "line": "There was a time I built things to prove I could. Now I build them so the people I love don't have to be afraid."
            },
            "elder": {
                "label": "Elder (68)",
                "voice_modifier": "gravelly, slower, occasional wry smile in the voice, hard-won peace, sarcasm softened into warmth",
                "line": "I used to think being the smartest person in the room was everything. Turns out, it's just the starting point."
            }
        }
    },
    "sam_cole": {
        "name": "Sam Cole",
        "char_id": "sam_cole",
        "archetype_note": "The Young Hero — scrappy neighborhood protector",
        "personality_core": "an earnest, awkward, genuinely good-hearted young person — idealistic and brave in a stumbling way, always trying to do the right thing",
        "ages": {
            "youth": {
                "label": "Youth (16)",
                "voice_modifier": "nervous teenage energy, slightly cracking voice, eager but scared, words rushing out faster than thoughts",
                "line": "I don't know if I'm ready. I'm probably not. But someone has to do something, so — yeah. It's going to be me."
            },
            "prime": {
                "label": "Prime (29)",
                "voice_modifier": "growing confidence, still warm and genuine, slight roughness from experience, earnest but no longer naive",
                "line": "I've learned that doing the right thing and doing the easy thing are almost never the same thing. I'm okay with that now."
            },
            "middle": {
                "label": "Middle (47)",
                "voice_modifier": "battle-worn but not broken, steady and measured, the kindness is still there but it's been tested, slight gravelly quality",
                "line": "I've lost people. Made mistakes I can't take back. But I never stopped believing it was worth trying. That has to count for something."
            },
            "elder": {
                "label": "Elder (66)",
                "voice_modifier": "soft and tired but at peace, slower delivery, warmth in every word, a mentor's calm",
                "line": "I've been scared every single time. Every single one. The trick — the only trick — is going anyway."
            }
        }
    },
    "elara_vex": {
        "name": "Elara Vex",
        "char_id": "elara_vex",
        "archetype_note": "The Dark Vigilante — broken warrior seeking justice",
        "personality_core": "a cold, precise, deeply wounded female warrior — emotion locked away, voice controlled and quiet, dangerous stillness beneath every word",
        "ages": {
            "youth": {
                "label": "Youth (19)",
                "voice_modifier": "raw grief barely held back, cold and clipped, the emotion bleeding through the attempted control, brittle",
                "line": "I don't need a reason. I don't need permission. I just need them to stop."
            },
            "prime": {
                "label": "Prime (31)",
                "voice_modifier": "perfectly controlled, dangerously quiet, every word chosen like a weapon, no wasted breath",
                "line": "Fear is a tool. Guilt is a tool. The only question that matters is what you're willing to do with them."
            },
            "middle": {
                "label": "Middle (49)",
                "voice_modifier": "slightly softer edges, the control still there but less rigid, exhaustion replacing rage, something like hope breaking through",
                "line": "I spent twenty years believing justice was something you take by force. I'm starting to think I had the word wrong."
            },
            "elder": {
                "label": "Elder (67)",
                "voice_modifier": "quiet and measured, warmth finally audible, still precise but no longer sharp, earned stillness",
                "line": "I spent thirty years in the dark. The light is unfamiliar. But I'm learning. Slowly. I'm learning."
            }
        }
    },
    "kael_ryn": {
        "name": "Kael Ryn",
        "char_id": "kael_ryn",
        "archetype_note": "The Chosen Rebel — space-faring hero with a destiny",
        "personality_core": "an idealistic, headstrong rebel with an almost mystical gift — passionate and impulsive in youth, growing toward a transcendent calm",
        "ages": {
            "youth": {
                "label": "Youth (18)",
                "voice_modifier": "breathless excitement, voice cracking with passion, wide-eyed idealism, impulsive and earnest",
                "line": "They told me the old ways were gone. That no one believed anymore. They were wrong. I can feel it. It's still there."
            },
            "prime": {
                "label": "Prime (30)",
                "voice_modifier": "hardened by battle but still passionate, controlled strength, charismatic leader's cadence, intensity without the recklessness",
                "line": "The cause isn't just a cause. It never was. It's every person who looked up and decided the galaxy could be different."
            },
            "middle": {
                "label": "Middle (50)",
                "voice_modifier": "slower and more deliberate, wisdom showing in the pauses, still magnetic but quieter, the weight of choices in every word",
                "line": "I've seen what absolute certainty does to people. I was certain once. I try to hold my convictions more carefully now."
            },
            "elder": {
                "label": "Elder (72)",
                "voice_modifier": "deeply calm, almost serene, sparse words given enormous weight, transcendent peace",
                "line": "I stopped fighting the galaxy. Now I try to understand it. That is harder. And more important."
            }
        }
    },
    "viktor_cross": {
        "name": "Viktor Cross",
        "char_id": "viktor_cross",
        "archetype_note": "The Super Soldier — man out of time, carrying impossible weight",
        "personality_core": "a principled, stoic, deeply loyal soldier — formal speech patterns, immense physical presence implied in voice, carries grief like armor",
        "ages": {
            "youth": {
                "label": "Youth (22)",
                "voice_modifier": "clear and earnest, soldier's cadence already present, idealistic belief in duty, unflinching but not yet hardened",
                "line": "I don't know what I'm fighting for yet. But I know it's worth it. I can feel that much. And that's enough to start."
            },
            "prime": {
                "label": "Prime (34)",
                "voice_modifier": "deep and commanding, quietly authoritative, immense controlled strength, grief underneath the resolve",
                "line": "I've watched the world change faster than I can follow. But some things don't change. And I hold onto those things."
            },
            "middle": {
                "label": "Middle (55)",
                "voice_modifier": "heavier, slower, the grief fully present now, still precise and controlled but the weariness is audible",
                "line": "Every generation thinks they'll be the last one to suffer like this. They're never right. But I keep hoping they will be."
            },
            "elder": {
                "label": "Elder (75)",
                "voice_modifier": "gravelly and slow, immense weariness, haunted quality, quiet dignity",
                "line": "I've outlived every enemy and half my friends. That's not victory. That's not even close to victory. That's just surviving."
            }
        }
    },
    "diana_fortes": {
        "name": "Diana Fortes",
        "char_id": "diana_fortes",
        "archetype_note": "The Warrior Goddess — immortal protector of humanity",
        "personality_core": "a fierce, noble, immortal warrior — speaks with the earnestness of someone who takes words seriously, powerful and precise, believes deeply in humanity",
        "ages": {
            "youth": {
                "label": "Youth (20)",
                "voice_modifier": "bright and fierce, absolutely certain, ringing clarity, noble formality not yet tempered by complexity",
                "line": "They told me this world wasn't worth saving. That humanity was too broken. They were wrong. I knew it then. I know it now."
            },
            "prime": {
                "label": "Prime (35 — appears ageless)",
                "voice_modifier": "commanding and resonant, warrior's authority, still passionate but tempered with decades of experience, compassionate strength",
                "line": "I fight not because I am certain of victory. I fight because it is right. The outcome does not change the obligation."
            },
            "middle": {
                "label": "Middle (appears 45, is ancient)",
                "voice_modifier": "deeper gravity, centuries of weight in the pauses, still fierce but with profound sadness beneath the power",
                "line": "I have watched empires rise and fall and rise again. The names change. The courage — the small, ordinary courage of ordinary people — never does."
            },
            "elder": {
                "label": "Elder (ancient — appears 60)",
                "voice_modifier": "vast and slow, an almost geological weight to each word, ancient tiredness and ancient love in equal measure",
                "line": "I have watched a thousand years of your wars. And still you find reasons to hope. That is why I stay. That has always been why."
            }
        }
    },
    "walter_marsh": {
        "name": "Walter Marsh",
        "char_id": "walter_marsh",
        "archetype_note": "The Fallen Everyman — ordinary man's tragic descent",
        "personality_core": "an ordinary, mild-mannered man whose voice transforms as he descends — starts timid and suppressed, ends cold and hollow",
        "ages": {
            "youth": {
                "label": "Youth (24)",
                "voice_modifier": "nervous, slightly defeated, voice that doesn't carry a room, earnest but unseen, slightly apologetic",
                "line": "I just want to do good work. Build something I can be proud of. Provide for my family. That's all I've ever wanted."
            },
            "prime": {
                "label": "Prime (40)",
                "voice_modifier": "still quiet but with a new flatness, the warmth draining away, calculated and careful, slight menace in the evenness",
                "line": "I made a choice. A very precise, deliberate choice. And I would make it again. That's what you need to understand."
            },
            "middle": {
                "label": "Middle (52)",
                "voice_modifier": "cold and hollow, no warmth left, the monster wearing a reasonable man's voice, chilling precision",
                "line": "You ask me if I feel anything. The honest answer is that I stopped asking myself that question a long time ago."
            },
            "elder": {
                "label": "Elder (62)",
                "voice_modifier": "broken and hollow, like a man finally seeing what he became, quiet devastation, the warmth gone and unmissed",
                "line": "I told myself it was for them. For years I told myself that. But somewhere along the way, it stopped being for anyone but me."
            }
        }
    },
    "james_holt": {
        "name": "James Holt",
        "char_id": "james_holt",
        "archetype_note": "The Suave Operative — charming spy, always the cleverest person present",
        "personality_core": "a charismatic, smooth, impossibly composed secret operative — voice like a well-made weapon, effortlessly charming, always in control",
        "ages": {
            "youth": {
                "label": "Youth (24)",
                "voice_modifier": "charming rogue energy, slight swagger, quick and playful, the polish is there but still a hint of roughness beneath",
                "line": "The name's Holt. And yes, before you ask — it's exactly as exciting as it sounds. Possibly more."
            },
            "prime": {
                "label": "Prime (38)",
                "voice_modifier": "perfectly polished, immaculate control, every word precisely weighted, warmth as a tool not a feeling, dangerous elegance",
                "line": "I've found that the most useful skill in my line of work isn't fighting or lying. It's listening. Everyone wants to be heard."
            },
            "middle": {
                "label": "Middle (52)",
                "voice_modifier": "the polish remains but something colder beneath, weariness around the edges, still controlled but the cost is showing",
                "line": "I have represented my country in ways my country will never know about. I made peace with that a long time ago. Mostly."
            },
            "elder": {
                "label": "Elder (69)",
                "voice_modifier": "gravel in the polish now, slower and more deliberate, the charm still there but genuine now, earned authority",
                "line": "I've lied to every government on three continents. Professionally. But I never lied to myself about what I was. That's the only line I didn't cross."
            }
        }
    }
}

print(f"✅ Loaded {len(CHARACTERS)} characters with {sum(len(c['ages']) for c in CHARACTERS.values())} total age stages")

In [ ]:
def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
        print(f"   🧹 VRAM freed — {free/1e9:.1f} GB available")

def make_silence(sr, duration_s=2.0):
    """Create a numpy array of silence."""
    return np.zeros(int(sr * duration_s))

def print_voice_card(char):
    name = char['name']
    note = char['archetype_note']
    bar = '━' * 50
    print(f"\n{bar}")
    print(f"  {name}")
    print(f"  {note}")
    print(bar)
    for stage_key, age in char['ages'].items():
        print(f"  [{age['label']}] {age['line'][:70]}...")
    print()

In [ ]:
print("⏳ Loading Qwen3-TTS-12Hz-1.7B-VoiceDesign...")
print("   (First run downloads ~8GB model weights — subsequent runs use cache)")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✅ Model loaded! VRAM: {used:.1f} GB used / {total:.1f} GB total")
else:
    print("✅ Model loaded on CPU")

In [ ]:
AGE_KEYS = ["youth", "prime", "middle", "elder"]
total_clips = len(CHARACTERS) * len(AGE_KEYS)
generated = 0
all_paths = {}  # char_id -> {age_key -> filepath}

print(f"🎬 Generating {total_clips} voice clips ({len(CHARACTERS)} characters × {len(AGE_KEYS)} ages)")
print("This will take several minutes on T4 GPU.\n")

for char_id, char in CHARACTERS.items():
    print_voice_card(char)
    all_paths[char_id] = {}
    
    for age_key in AGE_KEYS:
        age = char["ages"][age_key]
        generated += 1
        
        # Build the voice design prompt
        instruct = f"{char['personality_core']}, {age['voice_modifier']}"
        
        print(f"  [{generated}/{total_clips}] {char['name']} — {age['label']}")
        print(f"    Line: \"{age['line'][:60]}...\"")
        
        # Generate audio
        wavs, sr = model.generate_voice_design(
            text=age["line"],
            language="English",
            instruct=instruct
        )
        
        # Save
        filename = f"aging_{char_id}_{age_key}.wav"
        path = os.path.join(OUTPUT_DIR, filename)
        sf.write(path, wavs[0], sr)
        all_paths[char_id][age_key] = (path, sr)
        
        print(f"    💾 Saved: {filename}")
        display(Audio(path))
        print()

print(f"\n✅ All {total_clips} clips generated!")

In [ ]:
print("📖 Building life story montages (Youth → Prime → Middle → Elder)...\n")

for char_id, char in CHARACTERS.items():
    bar = '━' * 50
    print(f"\n{bar}")
    print(f"  {char['name']} — Life Story")
    print(f"  {char['archetype_note']}")
    print(bar)
    
    segments = []
    sr_val = None
    
    for age_key in AGE_KEYS:
        age = char["ages"][age_key]
        path, sr_val = all_paths[char_id][age_key]
        audio, _ = sf.read(path)
        segments.append(audio)
        segments.append(make_silence(sr_val, 2.0))  # 2s pause between ages
        print(f"  [{age['label']}] {age['line']}")
    
    # Remove last silence
    segments = segments[:-1]
    combined = np.concatenate(segments)
    
    montage_path = os.path.join(OUTPUT_DIR, f"aging_{char_id}_life_story.wav")
    sf.write(montage_path, combined, sr_val)
    
    duration = len(combined) / sr_val
    print(f"\n  🎧 Full life story ({duration:.1f}s):")
    display(Audio(montage_path))
    print()

print("✅ All life story montages complete!")

In [ ]:
print("🔀 Building cross-character comparison reels (all 8 chars at same age)...\n")

AGE_LABELS = {
    "youth": "Youth (all 8 characters at their youngest)",
    "prime": "Prime (all 8 characters at peak)",
    "middle": "Middle (all 8 characters at midlife)",
    "elder": "Elder (all 8 characters in old age)"
}

for age_key, age_label in AGE_LABELS.items():
    print(f"\n🎬 {age_label}")
    print("   Sequence: " + " → ".join([c['name'] for c in CHARACTERS.values()]))
    
    segments = []
    sr_val = None
    
    for char_id, char in CHARACTERS.items():
        path, sr_val = all_paths[char_id][age_key]
        audio, _ = sf.read(path)
        age_info = char["ages"][age_key]
        segments.append(audio)
        segments.append(make_silence(sr_val, 1.0))  # 1s between characters
        print(f"   • {char['name']} [{age_info['label']}]")
    
    segments = segments[:-1]
    reel = np.concatenate(segments)
    
    reel_path = os.path.join(OUTPUT_DIR, f"aging_comparison_{age_key}.wav")
    sf.write(reel_path, reel, sr_val)
    
    duration = len(reel) / sr_val
    print(f"\n   🎧 Comparison reel ({duration:.1f}s):")
    display(Audio(reel_path))

print("\n✅ All comparison reels complete!")

In [ ]:
import glob

print("="*55)
print("  CHARACTER VOICE AGING — GENERATION SUMMARY")
print("="*55)

total_duration = 0
total_files = 0

for char_id, char in CHARACTERS.items():
    clips = [all_paths[char_id][k][0] for k in AGE_KEYS]
    durations = []
    for p in clips:
        data, rate = sf.read(p)
        durations.append(len(data) / rate)
    total_char = sum(durations)
    total_duration += total_char
    total_files += len(clips) + 1  # clips + montage
    
    print(f"\n  {char['name']}")
    for age_key, dur in zip(AGE_KEYS, durations):
        label = char['ages'][age_key]['label']
        print(f"    {label:25s} {dur:5.1f}s")
    print(f"    {'Life Story Montage':25s} {total_char + 3*2:.1f}s")

print(f"\n{'─'*55}")
print(f"  Total clips    : {total_clips} core + 8 montages + 4 reels = {total_clips + 12} files")
print(f"  Total duration : ~{total_duration:.0f}s of core audio")
print("="*55)

## 🎬 Using These Voice Files

### In Game Engines
**Unity Timeline:**
- Use `AudioClip` assets for each `aging_<char>_<age>.wav`
- In Timeline, place the youth clip at the start of a flashback sequence and the elder clip at the end
- The `aging_comparison_*.wav` files work great as character-select screen audio

**Godot AnimationPlayer:**
- Import WAVs as AudioStream resources
- Chain them in AnimationPlayer tracks to sync with cutscene animations
- Use the `_life_story.wav` files for continuous playback during scrolling montages

### In Film / Animation
- Each `_life_story.wav` is a ready-to-use scratch audio track for an animatic showing character age progression
- The 2-second silence gaps between ages can be used as edit points

### Extending This
- Generate more lines per age by using the same `instruct` prompt — VoiceDesign will produce slightly different but tonally consistent results
- For perfect consistency across many lines, use the Design→Clone pipeline (see `14_Companion_Voice_Set_MVP.ipynb`): save one age's audio as a reference, then use `create_voice_clone_prompt` to clone that exact voice for unlimited additional lines

In [ ]:
import shutil
try:
    from google.colab import files
    print("📦 Creating zip archive...")
    shutil.make_archive("/content/character_aging_output", "zip", OUTPUT_DIR)
    size_mb = os.path.getsize("/content/character_aging_output.zip") / (1024*1024)
    print(f"✅ Archive: character_aging_output.zip ({size_mb:.1f} MB)")
    print("⬇️  Downloading...")
    files.download("/content/character_aging_output.zip")
except ImportError:
    print("Not in Colab — files saved to:", OUTPUT_DIR)
    print("Files:", sorted(os.listdir(OUTPUT_DIR)))